# 02 · Silver — Transacciones

Transforma `bronze.transacciones` en `silver.transacciones`: limpia, tipa, enriquece y **marca** los problemas de calidad sin descartar registros.

| | |
|---|---|
| **Origen** | `<catalog>.bronze.transacciones` |
| **Destino** | `<catalog>.silver.transacciones` y `<catalog>.silver.ref_feriados` |
| **Procesamiento** | Reconstrucción completa en cada corrida: idempotente por diseño (DT-012) |
| **Calidad** | Flags en `dq_flags`; solo se eliminan duplicados de `id_transaccion`, que siguen en Bronze |

| Regla | Criterio | Decisión |
|---|---|---|
| `DIA_INHABIL` | Fecha local en fin de semana o feriado → `fecha_operativa` = siguiente día hábil | DT-004 |
| Variantes MEP / Cable | `moneda = USD`: sufijo `D` → MEP, `C` → Cable; `simbolo_base` sin el sufijo | DT-005 |
| `PRECIO_FUERA_DE_RANGO` | Desvío mayor a 10x de la mediana del símbolo (mín. 5 operaciones, sin opciones) | DT-006 |
| `MONTO_EXTREMO` | Monto mayor al p99.9 de su tipo de instrumento | DT-006 |
| Tipo de instrumento | Reglas sobre símbolo y descripción; el resto se clasifica como `ACCION` | DT-011 |

**Prerrequisito:** `setup/04_silver.sql` y el notebook `01_bronze_ingesta`.

## Parámetros

In [ ]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.config import PipelineConfig
from iol_pipeline.silver import (
    DQ_RULES,
    FERIADOS_AR,
    PRICE_DEVIATION_THRESHOLD,
    build_silver,
    holidays_df,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
bronze_table = cfg.bronze_table("transacciones")
silver_table = cfg.silver_table("transacciones")
holidays_table = cfg.silver_table("ref_feriados")

print(f"Origen:   {bronze_table}")
print(f"Destino:  {silver_table}")
print(f"Feriados: {holidays_table}")

## Precondiciones

In [ ]:
for table, script in [(bronze_table, "setup/02_bronze.sql"), (silver_table, "setup/04_silver.sql"), (holidays_table, "setup/04_silver.sql")]:
    assert spark.catalog.tableExists(table), f"No existe la tabla {table}. Ejecutar {script}"

bronze_rows = spark.table(bronze_table).count()
assert bronze_rows > 0, f"{bronze_table} está vacía. Ejecutar 01_bronze_ingesta"
print(f"Filas en bronze: {bronze_rows}")

## Calendario de feriados
La lista vive en `src/iol_pipeline/silver.py` (única fuente de verdad) y se publica como tabla para que sea consultable.

In [ ]:
holidays_df(spark).writeTo(holidays_table).overwrite(F.lit(True))
display(spark.table(holidays_table).orderBy("fecha"))

## Transformación Bronze → Silver

In [ ]:
processed_at = datetime.now(timezone.utc).replace(tzinfo=None)
result = build_silver(spark, bronze_table, silver_table, processed_at)
display(spark.createDataFrame([result.to_dict()]))

## Reglas de calidad
Catálogo de reglas aplicadas (`DQ_RULES` en `src/iol_pipeline/silver.py`) y cuántas filas marcó cada una. Una fila puede tener más de un flag.

In [ ]:
silver = spark.table(silver_table)
total = silver.count()

flag_counts = {
    r["flag"]: r["filas"]
    for r in silver.select(F.explode("dq_flags").alias("flag")).groupBy("flag").agg(F.count("*").alias("filas")).collect()
}
display(spark.createDataFrame(
    [
        (rule.name, rule.description, rule.severity, rule.condition,
         flag_counts.get(rule.name, 0), round(100 * flag_counts.get(rule.name, 0) / total, 3))
        for rule in DQ_RULES
    ],
    "regla STRING, descripcion STRING, severidad STRING, condicion STRING, filas_marcadas LONG, pct DOUBLE",
))

In [ ]:
display(silver.groupBy("dq_status").agg(F.count("*").alias("filas")).withColumn("pct", F.round(100 * F.col("filas") / total, 2)).orderBy("dq_status"))

### Precios fuera de rango
Detalle por símbolo (sin datos de clientes). El umbral es un desvío mayor a `PRICE_DEVIATION_THRESHOLD` veces la mediana.

In [ ]:
print(f"Umbral: {PRICE_DEVIATION_THRESHOLD}x la mediana del símbolo")
display(
    silver.where(F.array_contains("dq_flags", "PRECIO_FUERA_DE_RANGO"))
    .groupBy("simbolo", "moneda", "tipo_instrumento", "precio_referencia")
    .agg(F.count("*").alias("operaciones"), F.min("precio").alias("precio_min"), F.max("precio").alias("precio_max"))
    .orderBy("simbolo")
)

### Días inhábiles
Operaciones en fin de semana o feriado y el día hábil al que se imputan.

In [ ]:
display(
    silver.where(~F.col("es_dia_habil"))
    .groupBy("fecha_local", F.date_format("fecha_local", "EEEE").alias("dia_semana"), "fecha_operativa")
    .agg(F.count("*").alias("operaciones"))
    .orderBy("fecha_local")
)

## Enriquecimiento
### Tipo de instrumento

In [ ]:
display(
    silver.groupBy("tipo_instrumento")
    .agg(F.count("*").alias("operaciones"), F.countDistinct("simbolo").alias("simbolos"))
    .withColumn("pct_operaciones", F.round(100 * F.col("operaciones") / total, 2))
    .orderBy(F.desc("operaciones"))
)

### Variantes de liquidación
`simbolos_con_base_operada`: variantes cuyo ticker base en ARS también aparece en el dataset. No es condición para clasificar la variante (DT-005), pero indica cuántas podrán usar la cotización del ticker base.

In [ ]:
ars_symbols = silver.where("moneda = 'ARS'").select(F.col("simbolo").alias("simbolo_base")).distinct().withColumn("base_operada", F.lit(True))
display(
    silver.select("simbolo", "simbolo_base", "tipo_liquidacion").distinct()
    .join(ars_symbols, "simbolo_base", "left")
    .groupBy("tipo_liquidacion")
    .agg(F.count("*").alias("simbolos"), F.count("base_operada").alias("simbolos_con_base_operada"))
    .orderBy("tipo_liquidacion")
)

### Horario de mercado
Proporción de operaciones dentro del horario de BYMA (días hábiles, 11 a 17 hs).

In [ ]:
display(
    silver.groupBy("es_dia_habil", "en_horario_mercado")
    .agg(F.count("*").alias("operaciones"))
    .withColumn("pct", F.round(100 * F.col("operaciones") / total, 2))
    .orderBy("es_dia_habil", "en_horario_mercado")
)

## Resultado
Se devuelve al orquestador como JSON.

In [ ]:
summary = result.to_dict()
print(json.dumps(summary, indent=2, default=str))
dbutils.notebook.exit(json.dumps(summary, default=str))